In [ ]:
!pip install unsloth

# 1 - Load the Model + Auto QLoRA

In [17]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "HuggingFaceTB/SmolLM2-135M-Instruct",
    max_seq_length = 2048,
    dtype = None, # auto-detect best type (bf16/fp16)
    load_in_4bit = True, # QLoRA (4-bit quant.)
)

print("Done!")

==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM2-135M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.
Done!


# 2 - LoRA Adapters

In [22]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # params = (input_dim × r) + (r × output_dim), bigger -> more parameters for training
    lora_alpha = 32, # Double of r
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # "embed_tokens", "lm_head"] -> input/output layers -> better vocabulary
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

model.print_trainable_parameters()

trainable params: 4,884,480 || all params: 139,399,488 || trainable%: 3.5039


# 3 - Prepare Dataset

In [25]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft[:300]"   # only first 300 samples
)

def format_sample(sample):
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize = False,
        add_generation_prompt = False
    )
    return {"text": text}

dataset = dataset.map(format_sample, remove_columns = dataset.column_names)
split = dataset.train_test_split(test_size = 0.1, seed = 42)

print(f"Train: {len(split['train'])} | Test: {len(split['test'])}")
print("\nSample:")
print(split['train'][0]['text'][:300])

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Train: 270 | Test: 30

Sample:
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Provide five feasible and eco-friendly methods for upgrading your kitchen to minimize your environmental impact on a daily basis. Consider covering topics such as reducing water usage, 


# 4 - Train

In [26]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    
    output_dir="/kaggle/working/output",
    
    num_train_epochs=5,
    
    per_device_train_batch_size=2,
    
    gradient_accumulation_steps=4,  
    
    learning_rate=2e-4,
    
    max_seq_length=2048,
    
    logging_steps=10,
    
    eval_strategy="epoch",

    save_strategy="epoch",
    
    load_best_model_at_end=True,
    
    fp16=not torch.cuda.is_bf16_supported(),
    
    bf16=torch.cuda.is_bf16_supported(),     
    
    optim="adamw_8bit",              
    
    warmup_ratio=0.03,
    
    gradient_checkpointing=True,   
    
    dataset_text_field="text",
    
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    args=training_args,
)


model.print_trainable_parameters()

# Train
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/270 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/30 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 270 | Num Epochs = 5 | Total steps = 170
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,884,480 of 139,399,488 (3.50% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.682612,1.922189
2,1.758917,1.884900
3,1.718056,1.872439
4,1.644583,1.867523
5,1.616068,1.866275


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=170, training_loss=1.6931965098661534, metrics={'train_runtime': 444.9527, 'train_samples_per_second': 3.034, 'train_steps_per_second': 0.382, 'total_flos': 1323371918794752.0, 'train_loss': 1.6931965098661534, 'epoch': 5.0})

# 5 - Save the models

In [27]:
# Save the LoRA adapter
model.save_pretrained("/kaggle/working/output/final")
tokenizer.save_pretrained("/kaggle/working/output/final")

# merge adapter INTO the base model
model.save_pretrained_merged(
    "/kaggle/working/output/merged",
    tokenizer,
    save_method="merged_16bit"
)

print("Saved!")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/kaggle/working/output/merged`: 100%|██████████| 1/1 [00:00<00:00,  2.93it/s]


Successfully copied all 1 files from cache to `/kaggle/working/output/merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:01<00:00,  1.82s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/output/merged`
Saved!


# 6 - Inference

In [31]:
from unsloth import FastLanguageModel

# Load fine-tuned model for inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/working/output/final",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# inference mode — disables dropout, 2x faster with Unsloth
FastLanguageModel.for_inference(model)

def generate_response(user_message, system_prompt="You are a helpful assistant."):
    messages = [
        {"role": "system",  "content": system_prompt},
        {"role": "user",    "content": user_message},
    ]
    
    # Format prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            temperature=0.7,      
            top_p=0.9,           
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)
print("Answare:\n")
print(generate_response("What is machine learning?"))
print("---" * 50)
print("Answare:\n")
print(generate_response("Explain loss functions simply."))

==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

HuggingFaceTB/SmolLM2-135M-Instruct does not have a padding token! Will use pad_token = <|endoftext|>.


Both `max_new_tokens` (=500) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answare:



Both `max_new_tokens` (=500) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Machine learning, also known as artificial intelligence or ML, is the process of using computers to learn from data and make predictions or decisions without being explicitly programmed or taught. It involves training models on vast amounts of data, which helps them improve their performance over time by adjusting parameters based on feedback received. Machine learning has become an essential tool in various fields such as science, engineering, finance, healthcare, and more. The process typically involves feeding large datasets into algorithms, where the model learns patterns and relationships within the data itself. Once trained, the algorithm can be used for tasks such as prediction, classification, recommendation systems, and predictive analytics. However, there are different types of machine learning techniques, including supervised learning, unsupervised learning, and reinforcement learning. Overall, it's been around since the 1950s but gained significant attention during the deve